## Programmazione implicita vs esplicita
Per far comunicare due componenti distribuite possiamo lavorare a livelli di astrazione differenti.  

Con la **programmazione di rete esplicita** lo sviluppatore usa direttamente strumenti come le socket (es. TCP o UDP) per gestire lo scambio di messaggi, dal collegamento col destinatario alla costruzione ed invio delle richieste, ricezione e interpretazione delle risposte.  
Questo approccio richiede di occuparsi direttamente di molti dei dettagli di comunicazione. 

L'alternativa a questo è utilizzare un **middleware di comunicazione** (collocato logicamente tra applicazione e sistema operativo). Il middleware offre un'unterfaccia al livello applicativo e permette di gestire operazioni comuni che altrimenti ogni app dovrebbe implementare da zero, nascondendo tutti i dettagli di comunicazione sottostanti.  
In tal modo **lo sviluppatore può concentrarsi sulla logica applicativa**.

Nel caso RPC (o di *RMI remote method invocations*, lo vedremo con la programmazione a oggetti), la comunicazione viene espressa tramite una chiamata simile a quella di una normale funzione come se fosse in locale, per questo si parla di **programmazione di rete implicita** (lo scamhio di msg esiste, ma è gestito dietro l'interfaccia).

## RPC

**RPC** sta per **Remote Procedure Call**: *un client chiede l'esecuzione di una procedura che però si trova presso un altro processo, tipicamente su una macchina remota*. 

L'idea generale è quella introdotta all'inizio: 
1. il client chiama la procedura e attende
2. il server riceve la richiesta e la esegue
3. il risultato viene restituito al client, che può riprendere l'esecuzione

<img src="imgs/rpc2.png" alt="rpc" width="300"/>

**Parametri e risultati attraversano la rete sotto forma di messaggi, che però non devono essere costruiti manualmente dal codice applicativo, ma sono gestiti dal middleware.**

es. il client chiama `service.add(3, 5)`: sembra interfaccia di chiamata locale, ma il calcolo è effettuato in server remoto.  

Ma allora l'obiettivo **è rendere l'accesso remoto simile all'accesso locale** --> bisogna affrontare problemi quali latenza, possibili perdite di comunicazione e guasti indipendenti (come visto nella parte sui fallimenti di comunicazione).

Moltissime realizzazioni di RPC: **Sun RPC** (C), **Java RMI**, **gRPC** (Google), **RPyC** (Python), **Apache Thrift** (multi-lingua), **Go** (che le supporta nativamente). Alcune di queste sono legate ad uno specifico linguaggio, altre consentono invece interazione tra componenti scritti in linguaggi diversi (es. gRPC). Di queste vedremo per ognuna un esempio, approfondendo poi Go.

### Client e Server Stub
Immaginiamo una chiamata locale `newlist = append(data, dbList)`.  
In questo caso, il chiamante passa i parametri alla procedura (mettendoli nella pila) e conserva dove si trova la prossima istruzione da eseguire in un indirizzo di ritorno. Quando la procedura termina, il programma riprende dall'istruzione successiva (assegnazione) assegnando il risultato a `newlist`.  
(NB perché tutto nella pila? perché le procedure possono essere annidate, quindi se ad esempio il programma chiama append allora append a sua volta può chiamare altre procedure come chiamante lavorando con la pila, che poi si riavvolgerà per riprendere le esecuzioni correttamente)

<img src="imgs/append.png" alt="append" width="200"/>

Quindi a questo punto ci si chiede **come ottenere un comportamento simile quando chiamante e procedura si trovano su macchine diverse**?  
Non basta trasferire il controllo tramite una pila condivisa, in quanto i processi hanno spazi di memoria comunque separati! Serve quindi un meccanismo che traduca la chiamata in comunicazione di rete.

Per risolvere il problema si introducono due intermediari, detti **stub** o **proxy**: **Client Stub** e **Server Stub**.

**Client Stub**

Il client chiama una funzione disponibile localmente, lo stub.  
Lo stub ha un'interfaccia identica a quella della funzione remota, ma invece di eseguire direttamente il lavoro richiesto:
- prepara i parametri
- costruisce la richiesta
- usa il sistema di comunicazione
- riceve e interpreta la risposta
- restituisce il risultato al chiamante

**Server Stub**

Sul server invece, lo stub:
- riceve la richiesta
- ricostruisce i parametri originali (MA sotto forma di un linguaggio da lui comprensibile)
- individua e chiama la procedura che implementa il servizio richiesto
- prepara e invia la risposta

<img src="imgs/stub.png" alt="stub" width="300"/>

##### Marshalling e Unmarshalling
Gli argomenti delle funzioni non possono essere inviati "così come sono" nella memoria del client: *devono essere rappresentate in una forma che il destinatario possa interpretare*.

Il **marshalling** prepara i dati della chiamata in forma trasferibile (es. codifica dei parametri nel formato concordato e inserimento nel messaggio).  
L'**unmarshalling** esegue l'operazione inversa, leggere la rappresentazione ottenuta e ricostruire i parametri originali MA sotto forma di oggetti o strutture dati del linguaggio del destinatario.

**Questi due passaggi sono necessari perché due macchine diverse possono usare rappresentazioni diverse dei dati** (es. big-endian vs little-endian, o rappresentazioni diverse di stringhe e strutture dati) **perché eseguiti su architetture o piattaforme diverse** (al di là del linguaggio di programmazione).

---
Passaggi RPC completi:  
**Client-Server**:
- il codice client chiama il **client stub**
- il client stub effettua **marshalling** dei parametri costruendo il msg di richiesta
- attraverso il SO e la rete il msg raggiunge il server
- il SO del server lo consegna al **server stub**

**Server-Client**:
- il server stub effettua **unmarshalling** dei parametri e chiama localmente la procedura applicativa
- la procedura esegue il lavoro e restituisce il risultato al server stub
- il server stub effettua **marshalling** del risultato preparando così la risposta
- la risposta attraversa la rete
- l'SO del client lo consegna al client stub
- il client stub effettua **unmarshalling** del risultato e lo restituisce alla procedura chiamante

<img src="imgs/rpc-steps.png" alt="rpc steps" width="500"/>

---

## RPC: Key Requirements e Implementation Challenges
Quindi per far sì che una RPC appaia come locale, il middleware deve:
- **Scambiare messaggi tra client e server** identificando procedura richiesta, trasportando parametri, associando ogni risposta alla chiamata corretta
- **Gestire l'etereogeneità dei dati**: come detto due macchine diverse possono rappresentare i dati in modi diversi, quindi serve marshalling/unmarshalling. Qui importante distinguere:
  - **serializzazione**: rappresentare dati o oggetti come sequenza di byte
  - **marshalling**: preparare complessivamete i dati necessari alla comunicazione. Quindi la serializzazione al più è parte del marshalling.
- **Gestire fallimenti ed errori**

Quindi le sfide a livello implementativo sono:
1. **Gestire l'etereogeneità dei dati** in particolare dovendo coordinare client e server anche in termini di protocollo di comunicazione (es. se usare TCP, UDP o entrambi?)
2. **Passaggio per riferimento**: se un parametro è un riferimento (indirizzo) a un oggetto in memoria locale, come fare a passarlo nel server?
3. **Semantica dei fallimenti**: per la gestione dei fallimenti ed errori, che semantica usiamo? (per RPC tipicamente at-least-once o at-most-once)
4. **Bind client to server**: abbiamo dato per scontato finora che il client sapesse dove inviare la richiesta, ma in realtà *dovremmo individuare l'endpoint del server* (i.e. info necessarie a raggiungerlo) come indirizzo e porta.

### Eterogeneità dei dati
Serve gestirla perché **client e server possono rappresentaere i dati in modi diversi**, es. little o big endian, dimensioni diverse di tipi (quanti bit per un intero etc..), codifica di caratteri (se ASCII o UTF-8 etc..), layout delle strutture in memoria (es. padding, allineamento...) --> non basta copiare la memoria grezza del client nel msg, serve una rappresentazione esplicita per la trasmissione, detta *wire encoding*.

Esistono 4 strategie principali di conversione utilizzate dai middleware RPC (sono comunque alternative generali, valide anche in altri contesti):
1. **Codifica indicata nel msg** per cui il mittente specifica la codifica e il destinatario converte di conseguenza. Il limite principale è messaggi più grandi;
2. **Conversione nel formato del destinatario**: il client stub produce direttamente il formato richiesto dal ricevente. *Il limite è che il mittente deve conoscere i formati degli altri sistemi*.
3. **Formato comune**: si decide apriori una codifica comune. Allora il mittente converte i dati nel formato comune e il ricevente riconverte il formato comune nel proprio locale. 
4. **Intermediario (broker)**: una componente specifica si occupa delle conversioni. Qui lo *svantaggio è che la componente può diventare collo di bottiglia/SPOF*.

Il secondo approccio non è ben scalabile: supponendo di avere $N$ formati diversi nella rete (uno per ogni nodo) --> ogni nodo deve essere in grado di convertire i propri dati in $N-1$ formati diversi --> $O(N^2)$ conversioni totali.

Con un formato comune al contrario si ottiene una soluzione **scalabile**: ogni sistema deve saper convertire solo dal proprio formato a quello comune e viceversa --> $O(N)$ conversioni totali in una rete di $N$ sistemi, ognuno con formati diversi.  
La soluzione con formato comune **riduce anche il coupling** dato che aggiungere un nuovo sistema non richiede insegnargli tutti i formati già esistenti.  
Per queste ragioni **la soluzione a formato comune è quella prevalente nelle implementazioni RPC**.

Queste quattro soluzioni si ricollegano direttamente a due pattern:
- **Proxy**: se localmente abbiamo client/server stub, allora la conversione verso il formato adatto avviene grazie a questi intermediari. Ricordiamo che questi proxy nascondono quindi i dettagli di comunicazione fornendo un'interfaccia all'utente simile a quella di una chiamata locale.
- **Broker**: direttamente collegato alla soluzione 4, in questo caso l'intermediario è un componente separato ai client e server. Il broker può occuparsi anche di instradamento, individuare i servizi etc.. (es. CORBA visto a ISW)

### Passaggio per riferimento
Prima di vedere come RPC gestisce il problema, si ricordano i tre metodi principali di passaggio dei parametri in una chiamata di funzione:
1. **Passaggio per valore**: la procedura riceve una *copia* del valore, e modificarla non modifica il dato originale del chiamante
2. **Passaggio per riferimento**: la procedura riceve un *riferimento* (es. indirizzo in memoria) al dato originale, quindi modificandolo si modifica anche il dato del chiamante
3. **Passaggio copy-restore**: la procedura chiamata riceve una copia del valore (*copy-in*), ma al termine della chiamata il valore modificato viene copiato sul dato originale del chiamante (*copy-out*).  
Quindi la procedura lavora su una copia, ma il risultato finale modifica effettivamente la memoria del chiamante. 

**Il copy-restore è quello che interessa RPC**: sappiamo che *non possiamo passare un riferimento in memoria locale direttamente a un server remoto* --> *si simula il passaggio per riferimento inviando una copia del dato tramite client stub --> il server stub lavora su quella copia --> una volta arrivato il risultato, il client stub lo sostituisce al dato originale in memoria locale*.  
Chiaramente per fare ciò è necessario che si sappia la dimensione del risultato da copiare in memoria, sia per client che per server.   
**E se la struttura contiene altri puntatori ancora?** Allora si procede ricorsivamente, copiando anche i dati puntati dai puntatori etc..



### Semantica dei fallimenti
Ricordiamo come **at-least-once** si adatti bene a operazioni idempotenti, mentre **at-most-once** un po' più complessso in quanto richiede riconoscimento duplicati e gestione dello stato delle richieste ma garantisce che se il server ha ricevuto la richiesta, allora la esegue al più una volta.

Exactly-once non lo contempliamo nei nostri casi di studio perché richiede meccanismi più complessi di persistenza, coordinamento e recupero --> ci concentreremo quindi su RPC che implementano o at-least-once o at-most-once.

### Binding: trovare il server
Prima di inviare la richeista, il client deve sapere **quale endpoint contattare**. Due possibilità:
1. **Binding Statico**: l'endpoint è noto prima dell'esecuzione e fissato nel programma o nella configurazione. Questo approccio è semplice in quanto non richiede meccanismo dinamico di discovery, ma poco trasparente e *poco flessibile* (se l'indirizzo cambia occorre aggiornare il riferimento hard coded, potrebbe essere necessario un nuovo deployment...)
2. **Binding Dinamico**: l'endpoint è individuato runtime --> ciò consente di cambiare server o scegliere una replica senza modificare necessariamente il codice applicativo del client, ma il costo è lavoro aggiuntivo per trovare il server in questione, maggiore *overhead*

Il **Binding Dinamico** distingue due fasi:
- **Naming**, stabilisce precisamente quale servizio/interfaccia vuole usare il client
- **Addressing**, stabilisce quale endpoint concreto contattare per quel servizio.

Quindi il nome del servizio indica l'identità logica di ciò che cerchiamo, mentre l'indirizzo identifica dove raggiungerne un'implementazione.  
Vedremo come nonostante la trasparenza sia migliore del caso statico, non è sempre completa: **alcuni framework richiedono comunque al programmatore di conoscere l'hostname o altri dettagli del server da contattare..**

Si parla poi di **Addressing Esplicito** se il client cerca il server  inviando una richiesta in broadcast/multicast e attendendo una risposta, mentre l'**Addressing Implicito** se il client *utilizza un **name server** (aka binder, directory service, registry service) per trovare l'endpoint*. 

NB. il name server non è necessariamente un componente separato, anzi tipicamente è un servizio che gira sullo stesso host del server.

<img src="imgs/rpc-binding.png" alt="rpc binding" width="300"/>

In figura accade quanto segue: il server registra il servizio e le info per raggiungerlo nel name server, il client stub interroga il name server che allora restituisce le info di binding --> il client stub contatta il server che permette di eseguire la richiesta specifica.

Il name server mantiene una **tabella di binding** che può supportare oltre che le registrazioni dei server anche routing, aggiornamento e rimozione dinamica delle voci in base a timeout o heartbeat.

Per ridurre l'overhead si può utilizzare **caching** evitando così di ripetere il lookup ad ogni invocazione.


### Other issues

**RPC Sincrone e Asincrone**:  

Il primo caso è quello classico e garantito praticamente da tutte le implementazioni di RPC: il chiamante attende il risultato prima di proseguire oltre la chiamata.

Il secondo caso è più difficile: il chiamante avvia l'operazione e continua le sue attività. Il risultato è gestito successivamente con gli strumenti previsti dal framework, es. un handler o una conferma della ricezione della richiesta da parte del server.

Inoltre ricorda che le RPC che consideriamo usano tutte **comunicazione transiente**, i.e. non forniscono al modello una coda persistente che conserva le richieste aspettando che un destinatario assente torni disponibile.

**La trasparenza RPC è limitata**:

Se è vero che l'interfaccia può assomigliare a una funzione locale, il comportamento non è proprio identico.  
Anzitutto in termini di **prestazioni** RPC aggiunge molti costi tra cui codifica/decodifica, copia dei dati, passaggi attraverso SO runtime e soprattutto trasmissione su rete (dei confronti in LAN mostrano valori dell'ordine di 0.1-1ms, contro i ns di una chiamata locale, milioni di volte peggio specie se si va fuori dalla LAN!)

**Fallimenti**:

Il client può non trovare il server, perdere la connessione o non ricevere risposta; e client e server possono fallire indipendentemente.  
Anche usare TCP non garantisce che la procedura venga eseguita e che il client ne conosca il risultato, dato che **l'affidabilità del trasporto non risolve da sola i guasti applicativi** (es. il server va in crash dopo aver spacchettato i dati). 

**Concorrenza e replicazione**:

Non tutti i framework gestiscono automaticamente richieste concorrenti/repliche del server --> alcuni server possono elaborare una sola richiesta per volta. 

**Sicurezza**:

Oltre a tutto ciò, il middleware offre protezioni adeguate? Es. Autenticazione (client e server sono chi dicono di essere), Autorizzazione (il client può eseguire la procedura richiesta?), Confidenzialità (un osservatore può leggere i msg?), Integrità (si riescono a rilevare alterazioni dei msg?) e **protezione da replay attacks** (un attaccante può intercettare e ritrasmettere un msg valido?)  
Affronteremo meglio questi tempi in framework specifici.

---

## IDL Interface Definition Language
**Ma come faccciamo a generare automaticamente gli stub e facciamo concordare client e server su interfacce comuni?**  
Del resto molti linguaggi importanti (C, C++) non hanno supporto RPC built-in...

Una soluzione comune a questo problema è l'uso di un **IDL Interface Definition Language**, questo descrive precisamente:
- nome del servizio
- operazioni disponibili
- parametri e relativi tipi di dato
- valori di ritorno 

L'IDL a tutti gli effetti descrive quindi il **contratto di servizio** (e non il codice che ne realizza poi il comportamento).

```text
es. 
    service KeyValueStore {
        i32 put(1: string key, 2: string value);
        string get(1: string key);
    }
```
sarebbe a dire che il servizio si chiama KeyValueStore e ha due operazioni: `put` che prende due stringhe key e value e restituisce un intero a 32 bit, e `get` che prende una stringa e restituisce una stringa.  
I numeri 1 e 2 sono identificatori dei parametri nella definizione dell'interfaccia, per dare il giusto ordine.

<img src="imgs/rpc-idl.png" alt="rpc idl" width="400"/>

In figura *in arancione il codice scritto dal developer* (qui con developer si intende lo sviluppatore dell'app che usa RPC e che realizza il programma client e/o il servizio server, non l'utente finale), *in blu quello generato automaticamente dal compilatore IDL*.  

Il developer scrive **la definizione IDL** (definizione dell'interfaccia del servizio remoto, i.e. quali operazioni il client può chiamare, parametri da passare e il tipo di risultato), il **codice client che invoca il servizio** e **le funzioni server che realizzano le operazioni**.  
Il compilatore IDL genera il supporto necessario secondo il framework: client e server stub, codice di conversione dei dati e header al supporto di comunicazione.

Il codice applicativo viene compilato assieme al supporto generato e alle librerie necessarie.  
Si distinguono due compilatori in figura; **compilatore IDL** che dalla descrizione del servizio genera il codice di supporto e il **compilatore di linguaggio** che dal codice applicativo + quello generato produce il codice eseguibile.

## RPC case studies
Vediamo "brevemente" Sun RPC, Java RMI, Python RPyC. Approfondiremo invece Go e gRPC.

### Sun RPC
Una delle prime implementazioni di RPC, sviluppata da Sun Microsystems (ora Oracle) per il linguaggio C negli anni '90, ma ancora utilizzato in quanto legacy per NFS (Network File System).

Cerchiamo di capire come funziona con un esempio volutamente semplice: **un client invia un numero a un server, e il server ne deve restituire il quadrato**.

Lo sviluppatore deve:
1. Descrivere l'interfaccia del servizio in un file IDL (qui con estensione `.x`): `square.x`
2. Usare il compilatore IDL `rpcgen` per generare il codice di supporto
3. Scrivere il programma client `client.c` che invoca il servizio remoto e la funzione che esegue il calcolo sul server `server.c` (che implementa la logica del servizio).

La comunicazione di rete sarà gestita dagli stub e dalle librerie RPC, senza doverla programmare direttamente nel codice applicativo.

Sun RPC risolve il problema della **eterogeneità dei dati** usando un **formato comune XDR (External Data Representation)**.  

XDR usa in particolare un **formato binario con typing implicito**, per cui vengono trasmessi i valori dei parametri senza indicarne il tipo. Ciò è possibile perché *client e server conoscono già i tipi grazie alla definizione dell'interfaccia IDL*.  
Il file `.x` in questo senso contiene proprio descrizione delle procedure e dei tipi di dato utilizzati, ed è a partire dal `.x` che rpcgen genera stub e le funzioni di conversione.

#### `square.x`

<img src="imgs/sunrpc-x.png" alt="sunrpc x" width="500"/>

`square_in` contiene il campo `arg1`, i.e. il numero su cui calcolare il quadrato mentre `square_out` contiene il campo `res1` (risultato).

Si ha poi una definizione a tre livelli: `SQUARE PROG` è il programma RPC, i.e. il servizio che raggruppa le diverse procedure (in questo caso una sola, la nostra d'esempio), `SQUARE_VERS` indica una versione dell'interfaccia di quel programma e `SQUAREPROC` è la definizione della procedura specifica.

A ciascuno dei tre lvl è associato un id numerico: numero di programma, di versione e di procedura --> servono per identificare al sistema RPC esattamente quale servizio, di quale versione e con quale operazione interagire.

La dichiarazione centrale è `square_out SQUAREPROC(square_in) = 1;`: significa che la procedura remota `SQUAREPROC` **riceve un dato di tipo `square_in` e restituisce un dato di tipo `square_out`**. Il numero 1 identifica la procedura all'interno della versione del servizio (in questo caso la versione 1).

Per convenzione tutti gli identifier sono scritti **in maiuscolo nel file IDL `.x`**.

Si osserva inoltre come ogni procedura abbia per convenzione **un solo argomento di input e di output**. Per aggirare questo si possono raccogliere più campi in una struct, come fatto in questo esempio (anche se poi il campo è comunque uno solo sia per input che per output, un long).

Una volta definito l'interfaccia IDL, allo sviluppatore non resta che scrivere **il `main()` del client** (che prepara l'input e chiama il servizio) e **l'implementazione della procedura sul server**, che effettuerà effettivamente il calcolo.  
(NB. ci stiamo ponendo sempre nello scenario dove lo sviluppatore lavora per un'azienda che vuole definire un'architettura distribuita via RPC, quindi è da definire sia client che server).

**In Sun RPC, il `main()` del server non è scritto dallo sviluppatore, ma generato automaticamente da `rpcgen` nel codice del server stub** --> è proprio il server stub a chiamare la funzione sul server: quando arriva una richiesta lo stub identifica l'operazione precisa grazie agli id e invoca la funzione applicativa corrispondente scritta dal developer.

#### `server.c`

<img src="imgs/sunrpc-server.png" alt="sunrpc server" width="500"/>

Il file include `square.h`, generato da `rpcgen`. Questo contiene, tra le altre cose, le definizioni dei tipi dichiarati nel file IDL `square.x`.

La funzione implementata sul server si chiama `squareproc_1_svc` e prende il nome secondo la convenzione: `squareproc` è il nome della procedura ma minuscolo, `_1` è la versione del programma e `_svc` indica che si tratta di una funzione di servizio (lato server).

La funzione riceve due parametri: `square_in *inp` è puntatore alla struct contenente l'input, che già è stata ricostruita nella memoria locale del server dallo stub secondo la logica copy-restore;  
e `struct svc_req *rqstp` che contiene informazioni sulla richiesta RPC fornite dal middleware (es. id della richiesta, id del programma, id della procedura etc.. in questo caso non li usiamo). 

Si fa poi il calcolo semplicemente come `inp->arg1 * inp->arg1` e si assegna il risultato al campo `res1` della struct di output.  
La funzione poi restituisce il puntature alla struct contenente il risultato `return(&out)`, con `out` dichiarata prima `static` perché **il server stub deve proteggere il risultato anche dopo che la funzione è terminata**, per convertirlo e mandarlo al client.

#### `client.c`

<img src="imgs/sunrpc-client1.png" alt="sunrpc client1" width="500"/>

<img src="imgs/sunrpc-client2.png" alt="sunrpc client2" width="500"/>

Il programma viene eseguito **indicando il nome della macchina server ed il numero**: `./square_client <server_hostname> <integer>` --> qui si osserva quanto la trasparenza RPC non sia completa... 

Nel codice le variabili principali sono `host` (hostname del server), `in` (struct nella quale preparare il numero (input) da inviare al server), `result` (puntatore alla struct di output che riceveremo dal server, contenente il risultato) e `clnt` (Handle RPC: oggetto di supporto usato dalla libreria RPC per gestire la comunicazione con il server).

La chiamata importante nel codice è `clnt = clnt_create(host, SQUARE_PROG, SQUARE_VERS, "tcp");`: `clnt_create` prepara il collegamento logico con il servizio RPC specificando hostname server, programma rpc specifico, la sua versione e il protocollo di trasporto da usare. Qui si usa TCP, ma Sun RPC permette anche UDP.  
(NB qui ancora non viene chiesto di calcolare il quadrato, si sta preparando invece l'handle con cui fare le chiamate successive. Del resto a `clnt.create` non stiamo passando il nome della singola procedura).

Poi il primo controllo verifica che sia stato possibile creare l'handle: `if (clnt == NULL)` --> se errore usa errore specifico della libreria RPC `clnt_pcreateerror(host)`.

Dopodiché `in.arg1 = atol(argv[2]);` converte il numero scritto sulla riga di comando da str a int e lo inserisce nella struct di input (ricordiamo `argv` contiene argomenti passati dalla riga di comando sotto forma di stringhe. Eseguendo `./square_client localhost 5` allora `argv[2]` è la stringa `"5"` e la converto in int con `atol()`).

La chiamata centrale vera e propria è infine `result = squareproc_1(&in, clnt);`: `squareproc_1` è la funzione generata automaticamente da `rpcgen` che implementa il client stub e rappresenta localmente la procedura remota.  
Riceve l'indirizzo locale della struct di input e l'handle `clnt` che identifica il servizio da contattare.

Sotto quest'ultima chiamata apparentemente semplice, il middleware:
1. converte l'input in XDR
2. invia la richiesta al server, attendendo la risposta (**SunRPC è sincrono e bloccante!**)
3. ricopia il risultato nella memoria del client

Utile in definitiva distinguere i nomi `squareproc_1` e `squareproc_1_svc`: il primo è la funzione generata dal compilatore IDL che avvia la call remota tramite stub, il secondo è l'effettiva funzione di servizio implementata sul server.

Riassumendo: 

<img src="imgs/riass.png" alt="sunrpc client server" width="500"/>

a partire dal file IDL `square.x` il compilatore `rpcgen` genera:
- `square.h`: contiene principalmente tipi, identificatori...
- `square_clnt.c`: client stub (contiene `squareproc_1` che invoca la procedura remota)
- `square_svc.c`: server stub, compreso il `main()` del server che riceve le richieste e invoca la funzione di servizio `squareproc_1_svc` implementata dallo sviluppatore.
- `square_xdr.c`: contiene le funzioni di conversione da e verso XDR per i tipi definiti nel file IDL.

A ciò lo sviluppatore aggiunge il proprio codice applicativo: `client.c` e `server.c`. Dopodiché il normale compilatore C combina i file necessari per produrre due eseguibili distinti **client** e **server**.

In generale da ricordare che `rpcgen` genera **tutti i file necessari per il supporto RPC**, mentre **lo sviluppatore scrive il comportamento dell'applicazione**. 

#### Come viene trovata la porta di un server
Nel client abbiamo indicato l'hostname del server, ma **non la porta di servizio**. Come viene trovata?

Sun RPC usa un servizio chiamato **port mapper** (o `rcpbind`) che **gira sulla macchina del server** ed è in ascolto sulla porta 111 (TCP e UDP).  
Il port mapper mantiene una tabella di binding che associa:
$$\text{(program number, version number, protocol)} \longrightarrow \text{port number}$$
Il client conosce quindi una porta fissa da interrogare (111) per scoprire effettivamente la porta sulla quale è disponibile il servizio richiesto.

<img src="imgs/portmapper.png" alt="port mapper" width="400"/>

Per la registrazione: all'avvio del server, il codice generato registra il servizio presso il port mapper locale tramite una funzione `svc_register()`, comunicando il programma, versione e info necessarie per raggiungerlo (protocollo e porta).

Quando poi il client esegue `clnt_create` --> il middleware contatta il port mapper del server e chiede su quale porta trovare quel programma e quella versione per il protocollo di trasporto scelto.  
Una volta ottenuta la prota, **il client comunica direttamente con il servizio RPC** (il port mapper non rappresenta un intermediario, ma solo un servizio di lookup).

es. il client interroga `serverUniversita:111`, il port mapper risponde dicendo che il servizio richiesto via TCP è sulla porta 49311, allora il client invia la richiesta direttamente a `serverUniversita:49311`.

Da osservare che l'hostname della macchina permette di ricavarne l'IP via DNS, il port mapper permette invece di ottenere la porta del servizio RPC.

Il binding inoltre viene effettuato una volta sola per l'handle client, poi si riutilizza senza dover interrogare il port mapper ad ogni invocazione (si parla di *early binding* nel senso che il binding viene fatto per la prima invocazione e poi riutilizzato per le successive).

**Recap dei principali aspetti di Sun RPC**:
- *Organizzazione*: un programma RPC può offrire più procedure e avere più versioni
- *Parametri*: un solo parametro di input e uno di output, usare le struct per raggruppare più campi
- *Trasporto*: TCP o UDP
- *Invocazione*: sincrona e bloccante
- *Concorrenza*: il server generato da `rpcgen` di base può gestire una sola richiesta alla volta
- *Stub*: codice visibile e generato automaticamente da `rpcgen`, non scritto dallo sviluppatore
- *Binding*: individuazione della porta via port mapper
- *Sicurezza*: inizialmente assente del tutto, poi aggiornamento con *Secure RPC*.
- *Semantica dei guasti*: Sun RPC garantisce **at-least-once** (basato quindi sulla ritrasmissione della richiesta in caso di timeout, ma non gestisce duplicati... sarebbe buono usarlo per operazioni non idempotenti).

Per il passaggio di parametri ricorda il meccanismo alla base di copy-restore (i dati passano per la rete tramite copie convertite in XDR, il risultato sarà copiato nella memoria del client simulando il passaggio per riferimento).



### 